# مناقصه‌یار · TenderYar v0.3 «Tender library + company matching» — Course 3 project notebook
**Goal:** 60 tenders in one **persistent Chroma library** with metadata filters (province, category, Jalali deadline, bond) and a **company profile → top-5 tenders with Persian reasons**.

| Step | Course 3 lesson it re-uses | Cells | Visible result |
|---|---|---|---|
| Setup | restore v0.2, install, tests | 1–2 | `28 passed` |
| s1 Similarity by hand + leaderboard | W1 similarity metrics, lab 1 | 3–4 | cosine/dot/L2 table · recall@1/@5 table |
| s2 Persistent library | W1 Chroma, W2 CRUD lab | 5–7 | `count()` = 60, survives restart |
| s3 Filters | W1 Chroma filtering | 8–9 | Isfahan-only results after a Jalali date |
| s4 Company matching | W2 recommender (food lab) | 10–11 | precision@5 table, full > semantic |
| s5 Gradio tabs | W2 RAG app | 12 | two tabs + screenshot |
| Done | git tag v0.3 | 13 | tag pushed |

**MODE:** `"gemini"` needs `GOOGLE_API_KEY` in Colab Secrets. `"offline"` uses character n-gram embeddings (no key) — same plumbing, lower quality.

## 0 · Setup — restore v0.2, add v0.3 files, run the tests

In [ ]:
# Cell 1 · install + restore the repo (Drive keeps chroma_db between sessions)
!pip install -q -U langchain-google-genai langchain-core langchain-text-splitters chromadb sentence-transformers gradio pandas numpy
import os, pathlib
from google.colab import drive
drive.mount("/content/drive")
HOME = pathlib.Path("/content/drive/MyDrive/tenderyar")       # put the project folder (unzipped) here once
if not HOME.exists():
    from google.colab import files
    up = files.upload()                                        # choose tenderyar_c3_cpNN.zip
    !unzip -q -o "{list(up)[0]}" -d /content/drive/MyDrive/
os.chdir(HOME); print(os.getcwd())

In [ ]:
# Cell 2 · key + MODE + tests
MODE = "gemini"            # or "offline"
if MODE == "gemini":
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
EMBED = "models/gemini-embedding-2" if MODE == "gemini" else "offline"
!python data/make_c3_data.py
!python -m pytest -q

## s1 · Similarity by hand, then by machine (Week 1)

In [ ]:
# Cell 3 · three tiny vectors: cosine vs dot vs L2
import numpy as np
V = {"آسفالت": np.array([1., 2., 0.]), "آسفالت ×۲": np.array([2., 4., 0.]), "آب": np.array([0., 1., 2.])}
q = V["آسفالت"]
for k, v in V.items():
    cos = q @ v / (np.linalg.norm(q) * np.linalg.norm(v))
    print(f"{k:12s} dot={q @ v:5.1f}  cos={cos:.3f}  L2={np.linalg.norm(q - v):.3f}")
# same direction, double length: cosine 1.000 but L2 2.236 and dot 10 → normalise before you use dot

In [ ]:
# Cell 4 · embedding leaderboard (20 query–passage pairs)
import json
from tenderyar.library.embed_bench import bench, STEmbeddings
from tenderyar.rag.index import HashEmbeddings, get_embeddings
pairs = [json.loads(l) for l in open("data/embed_pairs.jsonl", encoding="utf-8")]
models = {"offline HashEmbeddings": HashEmbeddings()}
if MODE == "gemini":
    models["models/gemini-embedding-2"] = get_embeddings("models/gemini-embedding-2")
    models["PartAI/Tooka-SBERT"] = STEmbeddings("PartAI/Tooka-SBERT")
    models["BAAI/bge-m3"] = STEmbeddings("BAAI/bge-m3")       # ~2.3 GB download; use a GPU runtime
lb = bench(models, pairs); lb

## s2 · A persistent Chroma library with metadata (Weeks 1–2)

In [ ]:
# Cell 5 · create the collection + upsert 60 tenders
from tenderyar.library import store, search as S, match as M
from tenderyar.library.cards import load_cards
from tenderyar.rag.splitter import persian_splitter
emb = get_embeddings(EMBED)
DB = f"chroma_db_{MODE}"                                       # on Drive → survives a runtime restart
coll = store.get_collection(store.get_client(DB))
cards, sp = load_cards("data/cards.jsonl"), persian_splitter()
for c in cards:
    store.upsert_tender(coll, c, sp.split_text(open(c["text_path"], encoding="utf-8").read()), emb)
print(coll.count(), "chunks"); coll.get(ids=["S001:0"], include=["metadatas"])["metadatas"][0]

In [ ]:
# Cell 6 · stats per province / category
import pandas as pd
st = store.stats(coll); print(st["chunks"], "chunks ·", st["tenders"], "tenders")
pd.DataFrame({"province": st["by_province"]}).T

In [ ]:
# Cell 7 · CRUD: update a deadline, delete one tender, re-open
print(store.update_meta(coll, "S001", deadline_jalali="1405/09/01"), "chunk(s) updated")
print(store.delete_tender(coll, "S002"), "chunk(s) deleted →", coll.count())
again = store.get_collection(store.get_client(DB)); print("re-opened:", again.count())
store.upsert_tender(coll, next(c for c in cards if c["id"] == "S002"), [open("data/synth/notices/S002.txt", encoding="utf-8").read()], emb)  # put it back

## s3 · Filters: «Isfahan road works, still open, bond ≤ 5 bn» (Week 1 filtering)

In [ ]:
# Cell 8 · build_where + semantic only vs semantic + filter
w = S.build_where(province="اصفهان", category="راه و ساختمان", after="1405/08/15", max_bond=5_000_000_000)
print(json.dumps(w, ensure_ascii=False, indent=1))
q = "روکش آسفالت محور"
a = [(h["tender_id"], h["meta"]["province"]) for h in S.search(coll, emb, q, k=5)]
b = [(h["tender_id"], h["meta"]["province"], h["meta"]["deadline_jalali"]) for h in S.search(coll, emb, q, where=w, k=5)]
print("semantic only:", a); print("with filter :", b)

In [ ]:
# Cell 9 · Jalali text input «بعد از ۱۵ آبان ۱۴۰۵»
w2 = S.build_where(province=["اصفهان", "یزد"], after="۱۵ آبان ۱۴۰۵")
print(w2["$and"][1])
for h in S.search(coll, emb, "مناقصه", where=w2, k=10): print(h["tender_id"], h["meta"]["province"], h["meta"]["deadline_jalali"])
print(S.search(coll, emb, q, where=S.build_where(province="اصفهان", max_bond=100_000_000)))   # [] → filter too strict

## s4 · Company matching — two-stage recommender (Week 2)

In [ ]:
# Cell 10 · hard filter reasons for 5 tenders
p = json.load(open("data/profiles/zagros.json", encoding="utf-8"))
for c in cards[10:15]:
    ok, why = M.hard_filter(p, c); print(c["id"], "✓" if ok else "×", "؛ ".join(why))

In [ ]:
# Cell 11 · top-5 with reasons + precision@5 for 3 variants
gold = {g["profile"]: set(g["relevant"]) for g in map(json.loads, open("data/match_gold.jsonl", encoding="utf-8"))}
rows = []
for v in ("semantic", "filter", "full"):
    ps = {k: M.precision_at_k([r["tender_id"] for r in M.match(json.load(open(f"data/profiles/{k}.json", encoding="utf-8")), coll, emb, cards, variant=v)], rel) for k, rel in gold.items()}
    rows.append({"variant": v, **ps, "avg": round(sum(ps.values()) / 3, 3)})
display(pd.DataFrame(M.match(p, coll, emb, cards))[["tender_id", "score", "reasons_fa"]])
pd.DataFrame(rows)        # paste into docs/RESULTS.md

## s5 · Library tabs in the Gradio app (Week 2)

In [ ]:
# Cell 12 · Gradio: «کتابخانه» + «پیشنهاد برای شرکت من»
os.environ["TY_EMBED"], os.environ["TY_DB"] = EMBED, DB
from tenderyar.apps.gradio_library import build
build().launch(share=True, debug=False)      # screenshot → docs/img/v0_3_gradio.png

## Done · tag v0.3

In [ ]:
# Cell 13 · commit + tag + push
!git add -A && git commit -qm "v0.3 Tender library + matching" && git tag v0.3
# !git push origin main --tags     (after: git remote add origin https://github.com/<you>/tenderyar.git)